<a href="https://colab.research.google.com/github/yair-ytshaki/FastText_Exploration/blob/main/FastText.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Demonstrate FastText in Hebrew**

Our main mission, and what these trials are based on, is the necessity to handle identity and update new classes of location in Hebrew. This notebook demonstrates the use of FastText for Hebrew word embeddings, showcasing how to load pre-trained models and calculate word similarities. It then introduces an 'Adaptive Centroid' method to classify and dynamically update word categories, even with typos or variations. Finally, it explores how FastText handles spaces and hyphens in Hebrew words, highlighting the impact of character-level n-grams on vector representation and similarity calculations

First, Let's try the English version of fasttext:

In [ ]:
import fasttext
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(repo_id="facebook/fasttext-en-vectors", filename="model.bin")
model = fasttext.load_model(model_path)

model.words
#['the', 'of', 'and', 'to', 'in', 'a', 'that', 'is', ...]
len(model.words)

In [ ]:
model.get_nearest_neighbors("bread", k=5)

[(0.5641006231307983, 'butter'),
 (0.48875734210014343, 'loaf'),
 (0.4491206705570221, 'eat'),
 (0.42444291710853577, 'food'),
 (0.4229326844215393, 'cheese')]

[(0.5641006231307983, 'butter'),
 (0.48875734210014343, 'loaf'),
 (0.4491206705570221, 'eat'),
 (0.42444291710853577, 'food'),
 (0.4229326844215393, 'cheese')]

Now, let's wget the he.300.bin version:

In [3]:
!pip install fasttext

!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.he.300.bin.gz
!gunzip cc.he.300.bin.gz

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.2-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.2-py3-none-any.whl (310 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4647421 sha256=2924ecd3fdb2d8715277dffd06536ba4f03610db93e38410915fee6dd5609473
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext
--2026-03-06 17:57:41--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.he.300.bin.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.225.143.99, 13.225.143.54, 13.225.143.122, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.225.143.99|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4507758

In [4]:
import numpy as np
import fasttext
model = fasttext.load_model("cc.he.300.bin")

print(model.get_nearest_neighbors("ירושלים"))

[(0.7114121317863464, 'ירושלים-'), (0.6643053889274597, 'בירושלים'), (0.663345217704773, 'ירושלי'), (0.6582450866699219, 'ירושליםכ'), (0.6577113270759583, 'ירושלים-מלחה'), (0.6539099812507629, 'ירושלים-אשדוד'), (0.6502159237861633, 'ירושלים-צור'), (0.6494978070259094, 'וירושלים'), (0.6469478011131287, 'ירושליםג'), (0.6464448571205139, 'ירושליםירושלים')]


a short test for the encoding (It should show a length of 7 and a series of \xd7 bytes):

In [ ]:
word = "ירושלים"
print(f"Length of string: {len(word)}")
print(f"Bytes: {word.encode('utf-8')}")

Length of string: 7
Bytes: b'\xd7\x99\xd7\xa8\xd7\x95\xd7\xa9\xd7\x9c\xd7\x99\xd7\x9d'


We will measure similarity between the input word and the target class by the Cosine Similarity:

In [ ]:
########################### Cosine Similarity #######################

def get_similarity(word1, word2, model):
    v1 = model.get_word_vector(word1)
    v2 = model.get_word_vector(word2)

    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

# Example:
score = get_similarity("ירושלים", "ירושליםםם", model)
print(f"Similarity score: {score:.4f}")

Similarity score: 0.6242


##**Method #1: "Adaptive Centroid"**

(trying to force the model to understand a "Sentence" or a "Typo" as a single unit)

**Building a "Class Repository" (Memory)**

Instead of comparing each new sample to all past samples (which would be very slow), we will store only the average vector of each class (the Centroid).

In [ ]:
# Each class stores its current vector and the number of samples seen
class_memory = {
    "Jerusalem": {"vector": model.get_word_vector("ירושלים"), "count": 1},
    "תל אביב": {"vector": model.get_word_vector("תל אביב"), "count": 1},
    "חיפה": {"vector": model.get_word_vector("חיפה"), "count": 1}
}

In [ ]:
alpha = 0.05  # Only 5% influence from the new sample

def classify_and_update(new_name, memory, threshold=0.8):
    # Turn the new name into vector
    new_vector = model.get_word_vector(new_name)

    best_match = None
    max_sim = -1

    # Cosine similarity to every class in the memory
    for class_name, class_vector in memory.items():
        sim = np.dot(new_vector, class_vector) / (np.linalg.norm(new_vector) * np.linalg.norm(class_vector))
        if sim > max_sim:
            max_sim = sim
            best_match = class_name

    # Decision Making
    if max_sim >= threshold:
        print(f"שיוך לקלאס קיים: '{best_match}' (דמיון: {max_sim:.2d})")
        # Incremental Update: implementation of ONLINE LEARNING method, the
        # class turned dynamic and shifts toward new variations
        # Inside the update logic:
        memory[best_match]["vector"] = (1 - alpha) * memory[best_match]["vector"] + alpha * new_vector
        memory[best_match]["count"] += 1
        return best_match
    else:
        print(f"זוהה קלאס חדש! '{new_name}' (דמיון מקסימלי היה רק {max_sim:.2d})")
        memory[new_name] = new_vector
        return new_name

In [ ]:
classify_and_update("תל-אביב", class_memory)

TypeError: unsupported operand type(s) for *: 'float' and 'dict'

In [ ]:
# Initialize Memory with a few base classes
# We use the raw model vectors as our starting point
class_memory = {
    "Jerusalem": {"vector": model.get_word_vector("ירושלים"), "count": 1},
    "Tel Aviv": {"vector": model.get_word_vector("תל אביב"), "count": 1}
}

In [ ]:
def test_stream(new_input, threshold=0.6, alpha=0.05):
    global class_memory
    new_vec = model.get_word_vector(new_input)

    best_match = None
    max_sim = -1

    # Calculate similarities
    for name, data in class_memory.items():
        # Cosine similarity formula
        sim = np.dot(new_vec, data["vector"]) / (np.linalg.norm(new_vec) * np.linalg.norm(data["vector"]))
        if sim > max_sim:
            max_sim = sim
            best_match = name

    print(f"Input: '{new_input}'")

    if max_sim >= threshold:
        print(f"✅ Match: '{best_match}' (Sim: {max_sim:.4f})")
        # Incremental update (EMA)
        class_memory[best_match]["vector"] = (1 - alpha) * class_memory[best_match]["vector"] + alpha * new_vec
        class_memory[best_match]["count"] += 1
    else:
        print(f"🆕 New Class Detected: '{new_input}' (Best was '{best_match}' with {max_sim:.4f})")
        class_memory[new_input] = {"vector": new_vec, "count": 1}
    print("-" * 30)

In [ ]:
# Test 1: Typo (Should match Jerusalem)
test_stream("ירושליםם")

# Test 2: Regional variation (Should likely match Jerusalem)
test_stream("ירושלים הבנויה")

# Test 3: Completely different (Should create a new class)
test_stream("חיפה")

# Test 4: Typo of the newly created class
test_stream("חיפהה")

Input: 'ירושליםם'
✅ Match: 'Jerusalem' (Sim: 0.6052)
------------------------------
Input: 'ירושלים הבנויה'
🆕 New Class Detected: 'ירושלים הבנויה' (Best was 'Jerusalem' with 0.5094)
------------------------------
Input: 'חיפה'
🆕 New Class Detected: 'חיפה' (Best was 'Jerusalem' with 0.5621)
------------------------------
Input: 'חיפהה'
🆕 New Class Detected: 'חיפהה' (Best was 'חיפה' with 0.5120)
------------------------------


The results are low, even after if we modified the treshold to be 0.6. It can said a lot, which is that basically we need to change our strategy. Let's keep the trashold back to the 0.8 - 0.95 range and create new function:

In [ ]:
def test_stream_v2(new_input, threshold=0.7, alpha=0.05):
    global class_memory

    # 1. Basic Cleaning
    clean_input = new_input.strip()

    # 2. Get Vector & Manually Normalize it (Unit Vector)
    raw_vec = model.get_word_vector(clean_input)
    new_vec = raw_vec / np.linalg.norm(raw_vec)

    best_match = None
    max_sim = -1

    for name, data in class_memory.items():
        # Ensure memory vector is also normalized
        mem_vec = data["vector"] / np.linalg.norm(data["vector"])

        # Since they are normalized, dot product IS the cosine similarity
        sim = np.dot(new_vec, mem_vec)

        if sim > max_sim:
            max_sim = sim
            best_match = name

    print(f"Input: '{new_input}'")
    if max_sim >= threshold:
        print(f"✅ Match: '{best_match}' (Sim: {max_sim:.4f})")
        # Update with normalized vector
        class_memory[best_match]["vector"] = (1 - alpha) * mem_vec + alpha * new_vec
        class_memory[best_match]["count"] += 1
    else:
        print(f"🆕 New Class: '{new_input}' (Best: '{best_match}' @ {max_sim:.4f})")
        class_memory[new_input] = {"vector": new_vec, "count": 1}
    print("-" * 20)

In [ ]:
# Test 1: Typo (Should match Jerusalem)
test_stream_v2("ירושליםם")

# Test 2: Regional variation (Should likely match Jerusalem)
test_stream_v2("ירושלים הבנויה")

# Test 3: Completely different (Should create a new class)
test_stream_v2("חיפה")

# Test 4: Typo of the newly created class
test_stream_v2("חיפהה")

Input: 'ירושליםם'
🆕 New Class: 'ירושליםם' (Best: 'Jerusalem' @ 0.6340)
--------------------
Input: 'ירושלים הבנויה'
✅ Match: 'ירושלים הבנויה' (Sim: 1.0000)
--------------------
Input: 'חיפה'
✅ Match: 'חיפה' (Sim: 1.0000)
--------------------
Input: 'חיפהה'
✅ Match: 'חיפהה' (Sim: 1.0000)
--------------------


In [ ]:
v1 = model.get_word_vector("ירושלים")
v2 = model.get_word_vector("ירושליםם")
print(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

0.60523045


Let's try this, and see if the trashold moved:

In [ ]:
def get_clean_vector(text, model):
    # FastText gets confused by leading/trailing whitespace in Hebrew
    clean_text = text.strip()
    vec = model.get_word_vector(clean_text)
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec

# Let's re-run your specific test
v1 = get_clean_vector("ירושלים", model)
v2 = get_clean_vector("ירושליםם", model)
sim = np.dot(v1, v2)
print(f"New normalized similarity: {sim:.4f}")

New normalized similarity: 0.6052


Didn't move.

In [ ]:
### Let's introduce normalization (vec / norm) ###

def get_smart_vector(text, model):
    words = text.split()
    if len(words) > 1:
        # Average the vectors of all words in the name
        vectors = [get_clean_vector(w, model) for w in words]
        avg_vec = np.mean(vectors, axis=0)
        return avg_vec / np.linalg.norm(avg_vec)
    return get_clean_vector(text, model)


def test_stream_v3(new_input, alpha=0.05):
    global class_memory
    new_vec = get_smart_vector(new_input, model)

    results = []
    for name, data in class_memory.items():
        mem_vec = data["vector"]
        sim = np.dot(new_vec, mem_vec)
        results.append((name, sim))

    # Sort by similarity
    results.sort(key=lambda x: x[1], reverse=True)
    best_name, best_sim = results[0]

    print(f"Input: '{new_input}'")

    # Dynamic Threshold: If it's the first class, or similarity is high enough
    if best_sim > 0.58: # Adjusted to your 0.605 results
        print(f"✅ Match: '{best_name}' (Sim: {best_sim:.4f})")
        # Update centroid
        class_memory[best_name]["vector"] = (1 - alpha) * class_memory[best_name]["vector"] + alpha * new_vec
        class_memory[best_name]["count"] += 1
    else:
        print(f"🆕 New Class: '{new_input}' (Best: '{best_name}' @ {best_sim:.4f})")
        class_memory[new_input] = {"vector": new_vec, "count": 1}
    print("-" * 20)

In [ ]:
# Test 1: Typo (Should match Jerusalem)
test_stream_v3("ירושליםם")

# Test 2: Regional variation (Should likely match Jerusalem)
test_stream_v3("ירושלים הבנויה")

# Test 3: Completely different (Should create a new class)
test_stream_v3("חיפה")

Input: 'ירושליםם'
🆕 New Class: 'ירושליםם' (Best: 'Jerusalem' @ 0.2890)
--------------------
Input: 'ירושלים הבנויה'
🆕 New Class: 'ירושלים הבנויה' (Best: 'ירושליםם' @ 0.4662)
--------------------
Input: 'חיפה'
🆕 New Class: 'חיפה' (Best: 'ירושלים הבנויה' @ 0.4272)
--------------------


Remove Space:

In [ ]:
# RESET MEMORY - Important to start clean!
class_memory = {
    "Jerusalem": {"vector": get_simple_vector("ירושלים", model), "count": 1},
    "Tel Aviv": {"vector": get_simple_vector("תל אביב", model), "count": 1}
}

In [42]:
def get_simple_vector(text, model):
    # Remove all spaces and just take the 'core' name for now to test
    clean_text = text.strip().replace(" ", "")
    clean_text = clean_text.strip().replace("-", "")

    print("clean_text:", clean_text)
    vec = model.get_word_vector(clean_text)
    return vec

def test_stream_v4(new_input, threshold=0.55):
    print("Proceeding:", new_input)
    global class_memory
    new_vec = get_simple_vector(new_input, model)

    best_name = None
    max_sim = -1

    for name, data in class_memory.items():
        v1 = new_vec
        v2 = data["vector"]

        # Pure Cosine Similarity calculation
        sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

        if sim > max_sim:
            max_sim = sim
            best_name = name

    print(f"Input: '{new_input}' -> Best Match: '{best_name}' (Score: {sim:.4f})")

    if sim >= threshold:
        print(f"✅ Success")
    else:
        print(f"🆕 New Class")
        class_memory[new_input] = {"vector": new_vec, "count": 1}

In [ ]:
print("Test 1 (Raw):", model.get_word_vector("ירושלים הבנויה")[:3])
print("Test 2 (No Space):", model.get_word_vector("ירושליםהבנויה")[:3])

Test 1 (Raw): [-0.01778244 -0.00191959  0.01837318]
Test 2 (No Space): [-0.02243225  0.00443037 -0.00012024]


So, FastText actually treats the space as a character that generates its own n-grams.

In [ ]:
# Test 1: Typo (Should match Jerusalem)
test_stream_v4("ירושליםם")

# Test 2: Regional variation (Should likely match Jerusalem)
test_stream_v4("ירושלים הבנויה")

# Test 3: Completely different (Should create a new class)
test_stream_v4("חיפה")

Input: 'ירושליםם' -> Best Match: 'Jerusalem' (Score: 0.2799)
🆕 New Class
Input: 'ירושלים הבנויה' -> Best Match: 'ירושליםם' (Score: 0.5807)
✅ Success
Input: 'חיפה' -> Best Match: 'Jerusalem' (Score: 0.3505)
🆕 New Class


SO, a typo like "םם" at the end can sometimes "pollute" the vector more than a single letter change would in English.

In [47]:
# Re-initializing class_memory with 'Tel Aviv' based on 'תל אביב'
class_memory = {
    "Jerusalem": {"vector": get_simple_vector("ירושלים", model), "count": 1},
    "Tel Aviv": {"vector": get_simple_vector("תל אביב", model), "count": 1}
}

clean_text: ירושלים
clean_text: תלאביב


In [48]:
print('--- Initial Class Memory ---')
for name, data in class_memory.items():
    print(f"Class: {name}, Count: {data['count']}")

# Now, use test_stream_v4 to add 'תל-אביב'.
# It should match 'Tel Aviv' and update its vector.
test_stream_v4("תל-אביב")

print('\n--- Class Memory After Adding "תל-אביב" ---')
for name, data in class_memory.items():
    print(f"Class: {name}, Count: {data['count']}")

# Let's test another variation for Tel Aviv, like 'תל-אביב-יפו'
test_stream_v4("תל-אביב-יפו")

print('\n--- Class Memory After Adding "תל-אביב-יפו" ---')
for name, data in class_memory.items():
    print(f"Class: {name}, Count: {data['count']}")

--- Initial Class Memory ---
Class: Jerusalem, Count: 1
Class: Tel Aviv, Count: 1
Proceeding: תל-אביב
clean_text: תלאביב
Input: 'תל-אביב' -> Best Match: 'Tel Aviv' (Score: 1.0000)
✅ Success

--- Class Memory After Adding "תל-אביב" ---
Class: Jerusalem, Count: 1
Class: Tel Aviv, Count: 1
Proceeding: תל-אביב-יפו
clean_text: תלאביביפו
Input: 'תל-אביב-יפו' -> Best Match: 'Tel Aviv' (Score: 0.5748)
✅ Success

--- Class Memory After Adding "תל-אביב-יפו" ---
Class: Jerusalem, Count: 1
Class: Tel Aviv, Count: 1


In [45]:
class_memory = {
    "Jerusalem": {"vector": get_simple_vector("ירושלים", model), "count": 1},
    "Tel Aviv": {"vector": get_simple_vector("תל אביב", model), "count": 1},
    "Haifa": {"vector": get_simple_vector("חיפה", model), "count": 1},
    "Beer Sheva": {"vector": get_simple_vector("באר שבע", model), "count": 1},
    "Kfar Saba": {"vector": get_simple_vector("כפר-סבא", model), "count": 1},
    "Tiberias": {"vector": get_simple_vector("טבריה", model), "count": 1}
}

clean_text: ירושלים
clean_text: תלאביב
clean_text: חיפה
clean_text: בארשבע
clean_text: כפרסבא
clean_text: טבריה


In [46]:
print('--- Initial Class Memory ---')
for name, data in class_memory.items():
    print(f"Class: {name}, Count: {data['count']}")

from collections import deque
locations = ["תל-אביב", "תל-אביב-יפו", "דבריה", "חיםה", "כפר סבר"]
locations_queue = deque(locations)

while locations_queue:
    item = locations_queue.popleft()  # Remove from front
    test_stream_v4(item)

    print('\n--- Class Memory After Treating Last Item:')
    for name, data in class_memory.items():
        print(f"Class: {name}, Count: {data['count']}")
print('\n')

--- Initial Class Memory ---
Class: Jerusalem, Count: 1
Class: Tel Aviv, Count: 1
Class: Haifa, Count: 1
Class: Beer Sheva, Count: 1
Class: Kfar Saba, Count: 1
Class: Tiberias, Count: 1
Proceeding: תל-אביב
clean_text: תלאביב
Input: 'תל-אביב' -> Best Match: 'Tel Aviv' (Score: 0.2857)
🆕 New Class

--- Class Memory After Treating Last Item:
Class: Jerusalem, Count: 1
Class: Tel Aviv, Count: 1
Class: Haifa, Count: 1
Class: Beer Sheva, Count: 1
Class: Kfar Saba, Count: 1
Class: Tiberias, Count: 1
Class: תל-אביב, Count: 1
Proceeding: תל-אביב-יפו
clean_text: תלאביביפו
Input: 'תל-אביב-יפו' -> Best Match: 'Tel Aviv' (Score: 0.5748)
✅ Success

--- Class Memory After Treating Last Item:
Class: Jerusalem, Count: 1
Class: Tel Aviv, Count: 1
Class: Haifa, Count: 1
Class: Beer Sheva, Count: 1
Class: Kfar Saba, Count: 1
Class: Tiberias, Count: 1
Class: תל-אביב, Count: 1
Proceeding: דבריה
clean_text: דבריה
Input: 'דבריה' -> Best Match: 'Jerusalem' (Score: 0.0258)
🆕 New Class

--- Class Memory After Tre